In [1]:
from assignment_2_2024.msg import PlanningAction, PlanningGoal

import time
import random
import pickle
import rospy
import actionlib


def time_goals(goals):
    times = []

    for (x, y) in goals:
        start = time.time()
        old_reached = reached_goals

        update_goal(x, y)
        rospy.loginfo(f"Setting the goal at ({x}, {y})")

        while time.time() - start < 60.0:
            if reached_goals > old_reached:
                times.append(time.time() - start)
                break
        else:
            times.append(0)
            
    return times

In [2]:
active_goal = False
not_reached_goals = 0

def update_goal(x, y):
    global active_goal, not_reached_goals
    goal = PlanningGoal()
    
    if active_goal:
        not_reached_goals = not_reached_goals + 1
    else:
        active_goal = True

    goal.target_pose.header.frame_id = "map"
    goal.target_pose.pose.position.x = x
    goal.target_pose.pose.position.y = y
    
    client.send_goal(goal, feedback_cb=read_feedback)

In [3]:
active_goal = False
latest_feedback = None
reached_goals = 0

def read_feedback(feedback):
    global active_goal, latest_feedback, reached_goals
    latest_feedback = feedback

    if (active_goal and feedback.stat == 'Target reached!'):
        reached_goals = reached_goals + 1
        active_goal = False
        goal = None

In [ ]:
rospy.init_node('action_client')
client = actionlib.SimpleActionClient('/reaching_goal', PlanningAction)

random.seed(10)

goals = [ (random.randrange(-9, 9, 1), random.randrange(-9, 9, 1)) for i in range(3) ]

with open("goals.pkl", "wb") as f:
    pickle.dump(goals, f)

times = time_goals(goals)

with open("times_a.pkl", "wb") as f:
    pickle.dump(times, f)

[INFO] [1747759302.671218, 6124.163000]: Setting the goal at (-8, 4)
[INFO] [1747759362.672548, 6133.773000]: Setting the goal at (6, -9)
